# Register S3

In [1]:
import boto3
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
ingest_create_athena_table_emotions_passed = False

In [3]:
%store -r ingest_create_athena_db_passed

In [4]:
try:
    ingest_create_athena_db_passed
except NameError:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not create the Athena Database.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")

In [5]:
print(ingest_create_athena_db_passed)

True


In [6]:
if not ingest_create_athena_db_passed:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not create the Athena Database.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")
else:
    print("[OK]")

[OK]


In [7]:
%store -r s3_private_path_wav

In [8]:
try:
    s3_private_path_wav
except NameError:
    print("*****************************************************************************")
    print("[ERROR] PLEASE RE-RUN THE PREVIOUS COPY TSV TO S3 NOTEBOOK ******************")
    print("[ERROR] THIS NOTEBOOK WILL NOT RUN PROPERLY. ********************************")
    print("*****************************************************************************")

In [9]:
print(s3_private_path_wav)

s3://sagemaker-us-east-1-218117716191/audio/Datasets/


## Import PyAthena & Pandas

In [10]:
from pyathena import connect
import pandas as pd

## Create Athena Table from Local Files

In [11]:
#Delete tables if they exist, to remove old creations
conn = connect(
    s3_staging_dir='s3://sagemaker-us-east-1-218117716191/athena/staging/',
    region_name='us-east-1'
)
cursor = conn.cursor()

# 1. Drop old versions
drop_statements = [
    "DROP TABLE IF EXISTS audio_emotions.cleaned_emotion_csv",
    "DROP TABLE IF EXISTS audio_emotions.cleaned_age_csv",
    "DROP TABLE IF EXISTS audio_emotions.cleaned_gender_csv",
]
for stmt in drop_statements:
    print("Running:", stmt)
    cursor.execute(stmt)

Running: DROP TABLE IF EXISTS audio_emotions.cleaned_emotion_csv
Running: DROP TABLE IF EXISTS audio_emotions.cleaned_age_csv
Running: DROP TABLE IF EXISTS audio_emotions.cleaned_gender_csv


## RUN ONLY ONCE to adjust tables location

In [17]:
# Adjust for directory
!aws s3 cp s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age.csv \
           s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age/cleaned_age.csv

!aws s3 cp s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion.csv \
           s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion/cleaned_emotion.csv

!aws s3 cp s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender.csv \
           s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender/cleaned_gender.csv

copy: s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age.csv to s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age/cleaned_age.csv
copy: s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion.csv to s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion/cleaned_emotion.csv
copy: s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender.csv to s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender/cleaned_gender.csv


In [ ]:
#Delete Originals
#aws s3 rm s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age.csv
#aws s3 rm s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion.csv
#aws s3 rm s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender.csv

In [12]:
#Confirm Data Schema of CSVs
age = pd.read_csv("s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age/cleaned_age.csv")        # or s3 path with s3fs
emotion = pd.read_csv("s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion/cleaned_emotion.csv")
gender = pd.read_csv("s3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender/cleaned_gender.csv")

print("AGE cols:", list(age.columns))
print("EMOTION cols:", list(emotion.columns))
print("GENDER cols:", list(gender.columns))

print("AGE dtypes:\n", age.dtypes)
print("EMOTION dtypes:\n", emotion.dtypes)
print("GENDER dtypes:\n", gender.dtypes)

AGE cols: ['Unnamed: 0.1', 'Unnamed: 0', 'meanfreq', 'sd', 'median', 'Q25', 'Q75', 'IQR', 'skew', 'kurt', 'sp.ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx', 'label']
EMOTION cols: ['Unnamed: 0.1', 'Unnamed: 0', 'X', 'meanfreq', 'sd', 'median', 'Q25', 'Q75', 'IQR', 'skew', 'kurt', 'sp.ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx', 'label']
GENDER cols: ['Unnamed: 0', 'meanfreq', 'sd', 'median', 'Q25', 'Q75', 'IQR', 'skew', 'kurt', 'sp.ent', 'sfm', 'mode', 'centroid', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', 'dfrange', 'modindx', 'label']
AGE dtypes:
 Unnamed: 0.1      int64
Unnamed: 0        int64
meanfreq        float64
sd              float64
median          float64
Q25             float64
Q75             float64
IQR             float64
skew            float64
kurt            float64
sp.ent          float64
sfm             flo

## Create the Athena tables with schema

In [11]:
staging = "s3://sagemaker-us-east-1-218117716191/athena/staging/"

conn = connect(region_name=region, s3_staging_dir=staging)

statements = [
    """
    CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_emotion_csv (
      row_id        int,
      row_id2       int,
      X             int,
      meanfreq      double,
      sd            double,
      median        double,
      Q25           double,
      Q75           double,
      IQR           double,
      skew          double,
      kurt          double,
      sp_ent        double,
      sfm           double,
      mode          double,
      centroid      double,
      meanfun       double,
      minfun        double,
      maxfun        double,
      meandom       double,
      mindom        double,
      maxdom        double,
      dfrange       double,
      modindx       double,
      label         string
    )
    ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
    WITH SERDEPROPERTIES (
      'separatorChar' = ',',
      'quoteChar'     = '"',
      'escapeChar'    = '\\\\'
    )
    LOCATION 's3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_emotion/'
    TBLPROPERTIES ('skip.header.line.count'='1','serialization.null.format' = '');
    """,
    """
    CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_age_csv (
      row_id        int,
      row_id2       int,
      meanfreq      double,
      sd            double,
      median        double,
      Q25           double,
      Q75           double,
      IQR           double,
      skew          double,
      kurt          double,
      sp_ent        double,
      sfm           double,
      mode          double,
      centroid      double,
      meanfun       double,
      minfun        double,
      maxfun        double,
      meandom       double,
      mindom        double,
      maxdom        double,
      dfrange       double,
      modindx       double,
      label         string
    )
    ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
    WITH SERDEPROPERTIES (
      'separatorChar' = ',',
      'quoteChar'     = '"',
      'escapeChar'    = '\\\\'
    )
    LOCATION 's3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_age/'
    TBLPROPERTIES ('skip.header.line.count'='1','serialization.null.format' = '');
    """,
    """
    CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_gender_csv (
      row_id        int,
      meanfreq      double,
      sd            double,
      median        double,
      Q25           double,
      Q75           double,
      IQR           double,
      skew          double,
      kurt          double,
      sp_ent        double,
      sfm           double,
      mode          double,
      centroid      double,
      meanfun       double,
      minfun        double,
      maxfun        double,
      meandom       double,
      mindom        double,
      maxdom        double,
      dfrange       double,
      modindx       double,
      label         string
    )
    ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
    WITH SERDEPROPERTIES (
      'separatorChar' = ',',
      'quoteChar'     = '"',
      'escapeChar'    = '\\\\'
    )
    LOCATION 's3://sagemaker-us-east-1-218117716191/audio/Datasets/cleaned_gender/'
    TBLPROPERTIES ('skip.header.line.count'='1','serialization.null.format' = '');
    """
]

cursor = conn.cursor()
for stmt in statements:
    print("Running:\n", stmt.splitlines()[1].strip())  # first line
    cursor.execute(stmt)


Running:
 CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_emotion_csv (
Running:
 CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_age_csv (
Running:
 CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.cleaned_gender_csv (


In [12]:
# Confirm each table was created
df_emotion = pd.read_sql(
    "SELECT * FROM audio_emotions.cleaned_emotion_csv LIMIT 5", conn
)
display(df_emotion)

df_age = pd.read_sql(
    "SELECT * FROM audio_emotions.cleaned_age_csv LIMIT 5", conn
)
display(df_age)

df_gender = pd.read_sql(
    "SELECT * FROM audio_emotions.cleaned_gender_csv LIMIT 5", conn
)
display(df_gender)

/tmp/ipykernel_252/3491415759.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_emotion = pd.read_sql(


,row_id,row_id2,x,meanfreq,sd,median,q25,q75,iqr,skew,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0,1,1,0.181338,0.060495,0.187476,0.126197,0.233586,0.107389,0.869088,...,0.181338,0.137742,0.023022,0.271186,0.777344,0.085938,6.226562,6.140625,0.116586,sad
1,1,2,2,0.186897,0.062260,0.195070,0.130847,0.243987,0.113140,1.191767,...,0.186897,0.121811,0.018412,0.271186,0.930339,0.085938,4.000000,3.914062,0.144983,sad
2,2,3,3,0.189102,0.062901,0.204945,0.131422,0.249978,0.118556,1.312690,...,0.189102,0.123758,0.083333,0.262295,0.332386,0.085938,0.625000,0.539062,0.334783,sad
3,4,5,5,0.183036,0.060051,0.174115,0.129949,0.236967,0.107017,1.096409,...,0.183036,0.128469,0.044693,0.258065,1.012019,0.085938,5.468750,5.382812,0.304910,sad
4,5,6,6,0.168793,0.057910,0.156266,0.116783,0.216326,0.099543,1.386837,...,0.168793,0.109720,0.022472,0.235294,0.228795,0.093750,0.750000,0.656250,0.306777,sad


/tmp/ipykernel_252/3491415759.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_age = pd.read_sql(


,row_id,row_id2,meanfreq,sd,median,q25,q75,iqr,skew,kurt,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,66,67,0.133338,0.069304,0.107668,0.089192,0.195267,0.106075,3.043456,13.694173,...,0.133338,0.121968,0.047337,0.277457,0.822656,0.0,4.687500,4.687500,0.076296,young
1,84,85,0.137433,0.058518,0.112037,0.092841,0.200079,0.107238,2.807995,12.776650,...,0.137433,0.111204,0.047151,0.277457,1.313384,0.0,6.046875,6.046875,0.135811,young
2,85,86,0.142227,0.065447,0.112242,0.093455,0.202909,0.109455,2.380899,9.942833,...,0.142227,0.118711,0.047013,0.275862,0.593750,0.0,6.539062,6.539062,0.096102,matured
3,87,88,0.133325,0.072849,0.113360,0.082861,0.203753,0.120892,1.904123,7.799218,...,0.133325,0.116200,0.047105,0.279070,0.424922,0.0,5.812500,5.812500,0.081880,young
4,88,89,0.130487,0.070407,0.113418,0.076098,0.196188,0.120089,1.820873,8.561101,...,0.130487,0.114802,0.047151,0.279070,0.198070,0.0,1.078125,1.078125,0.131579,matured


/tmp/ipykernel_252/3491415759.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_gender = pd.read_sql(


,row_id,meanfreq,sd,median,q25,q75,iqr,skew,kurt,sp_ent,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,66,0.133338,0.069304,0.107668,0.089192,0.195267,0.106075,3.043456,13.694173,0.929512,...,0.133338,0.121968,0.047337,0.277457,0.822656,0.0,4.687500,4.687500,0.076296,male
1,84,0.137433,0.058518,0.112037,0.092841,0.200079,0.107238,2.807995,12.776650,0.911080,...,0.137433,0.111204,0.047151,0.277457,1.313384,0.0,6.046875,6.046875,0.135811,male
2,85,0.142227,0.065447,0.112242,0.093455,0.202909,0.109455,2.380899,9.942833,0.936040,...,0.142227,0.118711,0.047013,0.275862,0.593750,0.0,6.539062,6.539062,0.096102,male
3,87,0.133325,0.072849,0.113360,0.082861,0.203753,0.120892,1.904123,7.799218,0.958362,...,0.133325,0.116200,0.047105,0.279070,0.424922,0.0,5.812500,5.812500,0.081880,male
4,88,0.130487,0.070407,0.113418,0.076098,0.196188,0.120089,1.820873,8.561101,0.969568,...,0.130487,0.114802,0.047151,0.279070,0.198070,0.0,1.078125,1.078125,0.131579,male


## RUN ONLY ONCE Feature Extraction From WAV to CSV

In [15]:
%pip install librosa --quiet
%pip install tqdm --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import librosa
import numpy as np
from pathlib import Path
import io
from tqdm import tqdm

In [17]:
s3 = boto3.client("s3")
#bucket = "sagemaker-us-east-1-218117716191"
prefix = "audio/Datasets/"   # no leading slash

In [ ]:
rows = []

# First, gather all wav keys so tqdm knows the total
keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].lower().endswith(".wav"):
            keys.append(obj["Key"])

# For debugging
#print("Total keys returned from S3:", len(keys))
#print("First 10 keys:", keys[:10])

for i, key in enumerate(tqdm(keys, desc="Processing WAVs")):
    resp = s3.get_object(Bucket=bucket, Key=key)
    data = io.BytesIO(resp["Body"].read())
    y, sr = librosa.load(data, sr=None)

    S = np.abs(librosa.stft(y))
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr).mean()
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr).mean()
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr).mean()
    zcr = librosa.feature.zero_crossing_rate(y).mean()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_means = mfcc.mean(axis=1)

    f0, _, _ = librosa.pyin(y, fmin=50, fmax=500)
    f0 = f0[~np.isnan(f0)]
    meanfun = f0.mean() if f0.size else 0.0
    minfun = f0.min() if f0.size else 0.0
    maxfun = f0.max() if f0.size else 0.0

    # Robust filename → label mapping
    fname = key.split("/")[-1]
    parts = fname.split("-")
    emotion_code = parts[2] if len(parts) > 2 else None
    folder_label = key.split("/")[-2]

    rows.append({
        "row_id": i + 1,
        "meanfreq": centroid,
        "sd": bandwidth,
        "median": rolloff,
        "Q25": zcr,
        "Q75": mfcc_means[0],
        "IQR": mfcc_means[1],
        "skew": mfcc_means[2],
        "kurt": mfcc_means[3],
        "sp_ent": mfcc_means[4],
        "sfm": mfcc_means[5],
        "mode": mfcc_means[6],
        "centroid": centroid,
        "meanfun": meanfun,
        "minfun": minfun,
        "maxfun": maxfun,
        "meandom": mfcc_means[7],
        "mindom": mfcc_means[8],
        "maxdom": mfcc_means[9],
        "dfrange": mfcc_means[10],
        "modindx": mfcc_means[11],
        "s3_key": key,
        "label": folder_label,
    })

df_wav = pd.DataFrame(rows)
df_wav.to_csv("Datasets/cleaned_emotion_from_wav.csv", index=False)

Processing WAVs:   1%|          | 82/12798 [01:47<4:43:19,  1.34s/it]

## Join Into A Single Pandas DataFrame

In [13]:
#Check layout of new csv
df_wav2 = pd.read_csv("Datasets/cleaned_emotion_from_wav.csv")
df_wav2.head()

,row_id,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp_ent,...,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,s3_key,label
0,1,6134.619097,4971.857653,11312.629132,0.068893,-578.9886,62.405308,-8.723742,13.807440,1.789457,...,77.050490,52.063108,121.700351,1.681229,-3.129482,-0.872467,-9.711343,-3.922256,audio/Datasets/Emotions/Angry/03-01-05-01-01-0...,Angry
1,2,4555.876353,4131.910812,8682.336957,0.060929,-588.2538,59.247963,-8.007299,4.542205,-1.324075,...,114.871283,50.289647,180.250093,2.055398,-7.919816,-6.678195,-1.995374,-2.865559,audio/Datasets/Emotions/Angry/03-01-05-01-01-0...,Angry
2,3,5470.014223,5320.017985,11139.045878,0.095745,-653.0620,87.556090,-0.483919,21.858513,4.684999,...,51.314313,50.000000,56.123102,3.482066,-10.200921,-5.455110,-2.633465,-4.095180,audio/Datasets/Emotions/Angry/03-01-05-01-01-0...,Angry
3,4,4850.116737,4208.027807,9083.984375,0.069765,-585.7502,52.763430,-10.848470,5.038836,-3.499608,...,119.660073,50.000000,172.110287,-1.513591,-9.520382,-9.057547,-5.041693,-4.515840,audio/Datasets/Emotions/Angry/03-01-05-01-01-0...,Angry
4,5,5791.790219,5195.901891,11145.209194,0.065825,-567.5401,77.566950,-8.457943,3.920733,1.946076,...,86.784408,58.438862,131.190810,-2.265315,-1.581761,-2.410354,-5.664477,-3.541209,audio/Datasets/Emotions/Angry/03-01-05-01-01-0...,Angry


### Discovery to combine datasets into one CSV

In [14]:
df_emotion_cols = set(df_emotion.columns)
df_wav2_cols    = set(df_wav2.columns)

print("Only in df_emotion:", df_emotion_cols - df_wav2_cols)
print("Only in df_wav2:",    df_wav2_cols    - df_emotion_cols)

Only in df_emotion: {'q25', 'q75', 'row_id2', 'iqr', 'x'}
Only in df_wav2: {'IQR', 'Q75', 's3_key', 'Q25'}


In [15]:
# Make all column names lowercase
df_emotion.columns = df_emotion.columns.str.lower()
df_wav2.columns    = df_wav2.columns.str.lower()

# Re-check differences
df_emotion_cols = set(df_emotion.columns)
df_wav2_cols    = set(df_wav2.columns)

print("df_emotion only:", df_emotion_cols - df_wav2_cols)
print("df_wav2 only:",    df_wav2_cols    - df_emotion_cols)

df_emotion only: {'row_id2', 'x'}
df_wav2 only: {'s3_key'}


In [16]:
df_emotion = df_emotion.rename(columns={"x": "s3_key_s"})
df_wav2    = df_wav2.rename(columns={"s3_key": "s3_key_s"})

In [17]:
# 1) Drop 'row_id2' from df_emotion
df_emotion = df_emotion.drop(columns=["row_id2"])

# 2) Append (concat) the two tables
df_all = pd.concat([df_emotion, df_wav2], ignore_index=True)

### Save combined df to csv for next file

In [18]:
df_all.to_csv("Datasets/Combined_Emotion_Data.csv", index=False)

In [28]:
#Check layout of new csv
df_all = pd.read_csv("Datasets/Combined_Emotion_Data.csv")
df_all.head()

,row_id,s3_key_s,meanfreq,sd,median,q25,q75,iqr,skew,kurt,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0,1,0.181338,0.060495,0.187476,0.126197,0.233586,0.107389,0.869088,2.863717,...,0.181338,0.137742,0.023022,0.271186,0.777344,0.085938,6.226562,6.140625,0.116586,sad
1,1,2,0.186897,0.062260,0.195070,0.130847,0.243987,0.113140,1.191767,3.878650,...,0.186897,0.121811,0.018412,0.271186,0.930339,0.085938,4.000000,3.914062,0.144983,sad
2,2,3,0.189102,0.062901,0.204945,0.131422,0.249978,0.118556,1.312690,4.589995,...,0.189102,0.123758,0.083333,0.262295,0.332386,0.085938,0.625000,0.539062,0.334783,sad
3,4,5,0.183036,0.060051,0.174115,0.129949,0.236967,0.107017,1.096409,3.680995,...,0.183036,0.128469,0.044693,0.258065,1.012019,0.085938,5.468750,5.382812,0.304910,sad
4,5,6,0.168793,0.057910,0.156266,0.116783,0.216326,0.099543,1.386837,5.031744,...,0.168793,0.109720,0.022472,0.235294,0.228795,0.093750,0.750000,0.656250,0.306777,sad


### Create Athena Database of Combined File

In [51]:
#Drop table to start fresh if needed

#Delete tables if they exist, to remove old creations
conn = connect(
    s3_staging_dir='s3://sagemaker-us-east-1-218117716191/athena/staging/',
    region_name='us-east-1'
)
cursor = conn.cursor()

# 1. Drop old versions
drop_statements = [
    "DROP TABLE IF EXISTS audio_emotions.combined_emotion_data",
]
for stmt in drop_statements:
    print("Running:", stmt)
    cursor.execute(stmt)

Running: DROP TABLE IF EXISTS audio_emotions.combined_emotion_data


In [52]:
#Confirm it is deleted
cursor.execute("SHOW TABLES IN audio_emotions")
print(cursor.fetchall())

[('cleaned_age_csv',), ('cleaned_emotion_csv',), ('cleaned_gender_csv',), ('emotions',)]


In [53]:
staging = "s3://sagemaker-us-east-1-218117716191/athena/staging/"

conn = connect(region_name=region, s3_staging_dir=staging)

create_combined_sql = """
CREATE EXTERNAL TABLE IF NOT EXISTS audio_emotions.combined_emotion_data (
  row_id     int,
  s3_key_s   string,
  meanfreq   double,
  sd         double,
  median     double,
  q25        double,
  q75        double,
  iqr        double,
  skew       double,
  kurt       double,
  sp_ent     double,
  sfm        double,
  mode       double,
  centroid   double,
  meanfun    double,
  minfun     double,
  maxfun     double,
  meandom    double,
  mindom     double,
  maxdom     double,
  dfrange    double,
  modindx    double,
  label      string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar'     = '"',
  'escapeChar'    = '\\\\'
)
LOCATION 's3://sagemaker-us-east-1-218117716191/audio/Datasets/Combined_Emotion_Data/'
TBLPROPERTIES ('skip.header.line.count'='1','serialization.null.format' = '');
"""

cursor = conn.cursor()
cursor.execute(create_combined_sql)

In [49]:
#Confirm it is created
cursor.execute("SHOW TABLES IN audio_emotions")
print(cursor.fetchall())

[('cleaned_age_csv',), ('cleaned_emotion_csv',), ('cleaned_gender_csv',), ('combined_emotion_data',), ('emotions',)]


In [55]:
df_emotion_combined = pd.read_sql(
    "SELECT * FROM audio_emotions.combined_emotion_data LIMIT 5", conn
)
display(df_emotion_combined)

/tmp/ipykernel_252/498248534.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_emotion_combined = pd.read_sql(


,row_id,s3_key_s,meanfreq,sd,median,q25,q75,iqr,skew,kurt,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label


## Review the New Athena Table in Glue Catalog

In [33]:
from IPython.core.display import display, HTML

region = "us-east-1"
database = "audio_emotions"

glue_url = (
    f"https://{region}.console.aws.amazon.com/glue/home"
    f"?region={region}#catalog:tab=databases;database={database}"
)

display(
    HTML(
        f'<b>Review <a target="_blank" href="{glue_url}">AWS Glue Catalog (audio_emotions)</a></b>'
    )
)

/tmp/ipykernel_252/2980283929.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


## Store Variables for the Next Notebooks

In [58]:
%store

Stored variables and their in-db values:
ingest_create_athena_db_passed             -> True
s3_file_path_wav                           -> 's3://sagemaker-us-east-1-218117716191/audio/Datas
s3_private_path_wav                        -> 's3://sagemaker-us-east-1-218117716191/audio/Datas
setup_dependencies_passed                  -> True
setup_s3_bucket_passed                     -> True


## Release Resources

In [59]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [1]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>